# Worksheet 3 — One Training Step: How the Network Learns

**Your project:** *teaching a neural network to play Atari Pong using nothing but the
screen.*

The worksheet version of `03_train_step.ipynb`. Worksheet 2 built a network that turns
numbers into a probability. It was never told whether its answers were any good, so it
never improved.

This supplies the missing link:

$$
\text{probability} \rightarrow \text{sampled action} \rightarrow \text{score it}
\rightarrow \text{loss} \rightarrow \text{change the weights}
$$

Everything stays on tiny fake tensors. No emulator, no real episodes — deliberately, so
every quantity is small enough to read with your eyes.

## 1. What you will learn

1. How to turn a probability into an actual choice, and how to score that choice.
2. Why the sign of a weight is the entire learning signal.
3. How to run one complete update, and how to prove the weights moved.
4. How a list of rewards becomes a per-step weight, by discounting backwards through time.
5. The Pong-specific subtlety: why credit must not leak across a scored point.
6. Why gradients are accumulated over several episodes instead of applied each time.

After this you will have seen every mechanism in the real training loop. What is left is
real pixels and patience.

## Running this notebook

**Locally**: the environment comes from the repo. `uv sync --locked`, then
`uv run jupyter lab`. The cell below does nothing.

**On Colab**: the kernel exists before the project does, so the cell below clones the
repo and installs what Colab is missing. Run it first.

It installs `gymnasium` and `ale-py` only. Colab already provides a working CUDA build
of torch, and replacing it with the exact locked one means downloading the entire CUDA
stack for no benefit to anything taught here. The README describes the stricter route
if you ever need bit-identical versions.


In [1]:
# Colab bootstrap. Does nothing when you run this notebook locally.
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # The -d test makes re-running this cell harmless.
    ![ -d /content/pong-pytorch ] || git clone -q https://github.com/piyushahuja/pong-pytorch.git /content/pong-pytorch
    %cd /content/pong-pytorch
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    !~/.local/bin/uv pip install -q --system --python {sys.executable} gymnasium ale-py
    !~/.local/bin/uv pip install -q --system --python {sys.executable} -e . --no-deps

print("python:", sys.version.split()[0])
if IN_COLAB:
    print("working directory:", os.getcwd())


python: 3.12.14


In [2]:
# Confirm the environment is what you think it is.
import torch
import gymnasium
import ale_py

print("torch:     ", torch.__version__)
print("gymnasium: ", gymnasium.__version__)
print("ale-py:    ", ale_py.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch:      2.14.0
gymnasium:  1.3.0
ale-py:     0.12.1
CUDA available: False


In [3]:
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Bernoulli

print(torch.__version__)
import pong        # PROJECT_ROOT: paths that work from anywhere


2.14.0


## 2. From a probability to a choice

Suppose the network outputs a single number $p \in (0,1)$ (Notebook 2’s sigmoid head).

A Bernoulli distribution treats $p$ as “chance of 1” (vs 0).

- `sample()` — draw 0 or 1
- `log_prob(action)` — how unsurprising that draw was under $p$ (natural log)

You will store `log_prob` for whatever you actually sampled.

In [4]:
p = torch.tensor(0.7)                    # pretend this came from a network
dist = Bernoulli(probs=p)

torch.manual_seed(0)
action = dist.sample()
lp = dist.log_prob(action)

print("p:", p.item())
print("sampled action:", action.item(), "(1 means 'option A', 0 means 'option B')")
print("log_prob:", lp.item())
print("prob of that action:", lp.exp().item())

p: 0.699999988079071
sampled action: 1.0 (1 means 'option A', 0 means 'option B')
log_prob: -0.3566749393939972
prob of that action: 0.699999988079071


Check the arithmetic, because it is what makes this concrete rather than magical.

If the sample came out `1.0`, the probability of that draw was $p = 0.7$ and
$\log 0.7 \approx -0.357$. If it came out `0.0`, the probability was $0.3$ and
$\log 0.3 \approx -1.204$. The last line recovers the probability by exponentiating, and it
should match one of those exactly.

Note `log_prob` is always **negative**, because probabilities are below 1. A log-prob near
zero means "that was nearly certain"; a very negative one means "that was a surprise".

Several steps → several log-probs. Stack them into one tensor.

In [5]:
torch.manual_seed(1)
ps = torch.tensor([0.2, 0.5, 0.9])
dist = Bernoulli(probs=ps)
actions = dist.sample()
log_probs = dist.log_prob(actions)

print("p:        ", ps)
print("actions:  ", actions)
print("log_probs:", log_probs)
print("stacked shape:", log_probs.shape)

p:         tensor([0.2000, 0.5000, 0.9000])
actions:   tensor([0., 1., 1.])
log_probs: tensor([-0.2231, -0.6931, -0.1054])
stacked shape: torch.Size([3])


Match the columns up by hand. Where `p` was 0.9 and the action came out 1, the log-prob is
close to zero — a likely event. Where `p` was 0.2 and the action came out 1, it is much
more negative, because a 20% event happened.

This is the raw material of policy-gradient learning: for each decision, one number saying
how surprising the taken action was under the policy that took it.

## 3. Weight those log-probs

Idea: if a sequence of choices looked good, make those choices a bit more probable; if bad, a bit less.

A simple scalar loss that does that (gradient descent *minimizes* loss):

$$
L = -\sum_t w_t \, \log \pi(a_t)
$$

Here $w_t$ are made-up weights (later: discounted returns). Positive $w$ pushes up $\log\pi$; negative $w$ pushes it down.

In [6]:
weights = torch.tensor([1.0, -0.5, 2.0])   # fake "how good was this step?"
loss = -(log_probs * weights).sum()
print("loss:", loss.item())

loss: 0.08729106187820435


The middle entry is the one to look at. Its log-prob is negative and its weight is negative,
so their product is **positive**, while the other two products are negative. After the
overall minus sign, that middle action is the only one contributing to *increasing* the
loss.

Minimising this loss therefore pushes the first and third actions' probabilities up and the
middle one's down — entirely from the sign of $w$.

The loss value itself means nothing on its own. It is not an error, it is not bounded, and
it can be negative. Only its gradient matters.

## 4. One real update: forward → loss → backward → step

Wire a tiny net into the same pattern.

1. `y = model(x)` → sigmoid → $p$
2. sample + `log_prob`
3. build `loss`
4. `loss.backward()` writes gradients into `.grad`
5. `optimizer.step()` changes weights
6. `optimizer.zero_grad()` clears grads for next time

In [7]:
class TinyBinaryNet(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=16):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        return torch.sigmoid(self.layer2(x)).squeeze(-1)


torch.manual_seed(0)
model = TinyBinaryNet()
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-2)

x = torch.randn(8)
w_before = model.layer2.weight.detach().clone()

p = model(x)
dist = Bernoulli(probs=p)
action = dist.sample()
log_prob = dist.log_prob(action)

# pretend this choice was "good"
weight = torch.tensor(1.0)
loss = -(log_prob * weight)

optimizer.zero_grad()
loss.backward()
optimizer.step()

w_after = model.layer2.weight.detach()
print("p:", p.item(), "action:", action.item(), "loss:", loss.item())
print("layer2 weight changed:", not torch.allclose(w_before, w_after))
print("max |ΔW|:", (w_after - w_before).abs().max().item())

p: 0.4906153380870819 action: 1.0 loss: 0.7120949029922485
layer2 weight changed: True
max |ΔW|: 0.09999997913837433


The interesting part is what the printout proves about the mechanism.

Before `backward()`, there is no gradient at all — not zero, absent. Nothing has asked for a
derivative yet. After it, every weight has its own number saying how the loss would respond
to changing it. After `step()`, the weights have actually moved.

Look at the size of that movement. It is around 0.1 against a learning rate of 0.01 — ten
times larger. That is not a bug: RMSprop divides each gradient by the square root of a
running average of squared gradients, and on the very first step that average is tiny, so
the division inflates the step. Later steps, once the average has warmed up, land much
closer to the bare learning rate.

Also note we simply asserted the choice was good by passing `1.0`. The network believes what
we tell it. Getting those weights right is the real problem, and it is what the rest of this
worksheet is about.

## 5. Discounted returns from a reward list

Weights $w_t$ often come from future rewards. Walking backward:

$$
R_t = r_t + \gamma R_{t+1}
$$

with $R_{T+1} = 0$. So early steps feel the later $+1$ / $-1$, faded by $\gamma$.

In [8]:
def discount_rewards(rewards, gamma=0.99):
    """rewards: list of floats, e.g. [0, 0, 0, 1]."""
    discounted = [0.0] * len(rewards)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = running * gamma + rewards[t]
        discounted[t] = running
    return torch.tensor(discounted, dtype=torch.float32)


rewards = [0.0, 0.0, 0.0, 1.0]
returns = discount_rewards(rewards, gamma=0.99)
print("rewards:", rewards)
print("returns:", returns.tolist())

rewards: [0.0, 0.0, 0.0, 1.0]
returns: [0.9702990055084229, 0.9800999760627747, 0.9900000095367432, 1.0]


This output is the answer to "which action gets the credit?".

The reward arrived only at the last step, yet every earlier step now has a **nonzero**
weight, fading the further back you go. So an action taken 50 frames before a point still
receives $0.99^{50} \approx 0.6$ of the credit; at 300 frames it is
$0.99^{300} \approx 0.05$, which is almost nothing.

That is what $\gamma$ really controls — not a philosophical preference for the present, but
how long the memory is.

Often we normalize returns (zero mean, unit variance) so updates are not dominated by raw scale.

In [9]:
def normalize(returns):
    return (returns - returns.mean()) / (returns.std(unbiased=False) + 1e-8)


normed = normalize(returns)
print("normalized:", normed.tolist())
print("mean~", normed.mean().item(), "std~", normed.std(unbiased=False).item())

normalized: [-1.3371336460113525, -0.4516901671886444, 0.44270288944244385, 1.3461263179779053]
mean~ 1.341104507446289e-06 std~ 0.9999990463256836


Something important happened that is easy to miss: **some weights are now negative.**

Before normalising, every return was positive, because a point was won. After normalising,
the below-average steps became negative. So the update pushes the early actions' probabilities
*down* and the last one's *up*.

That is the point. Without it, every action in a winning episode is reinforced equally and
the network learns little about *which* actions mattered. Normalising converts absolute credit
into relative credit: better than average, or worse.

The `+ 1e-8` guards against dividing by zero when every return is identical.

### Optional: reset the running return on nonzero rewards

In some games a nonzero reward ends a *sub*-episode (a point). You can zero the running return there so credit does not leak across points.

In [10]:
def discount_rewards_reset_on_score(rewards, gamma=0.99):
    discounted = [0.0] * len(rewards)
    running = 0.0
    for t in reversed(range(len(rewards))):
        if rewards[t] != 0:
            running = 0.0
        running = running * gamma + rewards[t]
        discounted[t] = running
    return torch.tensor(discounted, dtype=torch.float32)


rewards2 = [0.0, 0.0, 1.0, 0.0, 0.0, -1.0]
print("plain :", discount_rewards(rewards2).tolist())
print("reset :", discount_rewards_reset_on_score(rewards2).tolist())

plain : [0.029109949246048927, 0.029403990134596825, 0.029701000079512596, -0.9800999760627747, -0.9900000095367432, -1.0]
reset : [0.9800999760627747, 0.9900000095367432, 1.0, -0.9800999760627747, -0.9900000095367432, -1.0]


Compare the two lines at the positions **before and including the won point**.

In `plain`, the later `-1` leaks backwards and very nearly cancels the `+1`, so actions that
led to a point being *won* score as barely better than neutral. The model is told they hardly
mattered, because something bad happened a few frames later in a rally they had no influence
over.

In `reset`, those same steps keep their full credit, because the `+1` stops the backward
flow. Each rally is scored on its own merits.

This is one line of code — `if rewards[t] != 0: running = 0.0` — and it encodes a real fact
about the game. That is what domain knowledge usually looks like in practice: not a clever
architecture, but knowing where to cut a sum.

## 6. Put it together (still fake data)

One “episode”: several forward passes, collect log-probs and rewards, discount, one backward.

In [11]:
torch.manual_seed(2)
model = TinyBinaryNet()
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-2)

log_probs = []
rewards = []

for t in range(5):
    x = torch.randn(8)
    p = model(x)
    dist = Bernoulli(probs=p)
    action = dist.sample()
    log_probs.append(dist.log_prob(action))
    # fake rewards: last step gets +1
    rewards.append(1.0 if t == 4 else 0.0)

returns = normalize(discount_rewards(rewards))
episode_log_probs = torch.stack(log_probs)

loss = -(episode_log_probs * returns).sum()

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("rewards:", rewards)
print("returns (norm):", returns.tolist())
print("loss:", loss.item())

rewards: [0.0, 0.0, 0.0, 0.0, 1.0]
returns (norm): [-1.407097339630127, -0.7106208801269531, -0.0071107009425759315, 0.7035101652145386, 1.4213058948516846]
loss: -0.025937378406524658


Read the normalised returns against the rewards. Only the last step earned anything, yet all
five weights are nonzero: the steps nearest the reward come out positive and the earliest
ones negative. One `backward()` call then adjusts every weight in the network, taking all
five decisions into account at once.

Compare this cell with the real training loop and there are only three differences: the input
would be a 6400-long preprocessed frame instead of 8 random numbers, the rewards would come
from ALE instead of a hand-written `if`, and the loop would run for thousands of frames rather
than five. The mechanism is identical.

## 7. Optional reading: same shapes as Notebook 2’s game examples

Still no environment — only “if $p$ came from that kind of head…”.

| Shape idea | $p$ / logits | Sampling |
|---|---|---|
| Pong-like | one sigmoid $p$ | `Bernoulli(p)` → 0/1 |
| Chess / Go-like | many logits → softmax | sample a category (not shown here) |

This notebook practiced the Bernoulli path because it matches a single probability output.

In [12]:
# Pong-shaped head: 6400 -> 1 probability (fake frame)
pong_head = nn.Sequential(
    nn.Linear(6400, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
    nn.Sigmoid(),
)
fake_frame = torch.randn(6400)
p = pong_head(fake_frame).squeeze()
action = Bernoulli(probs=p).sample()
print("pong-like p", p.item(), "sample", action.item())

pong-like p 0.5982619524002075 sample 0.0


In [13]:
# Closing the loop back to Worksheet 1: a probability becomes a controller input.
from pong.env import ACTION_UP, ACTION_DOWN

print("ALE action would be:", ACTION_UP if action.item() == 1 else ACTION_DOWN,
      f"({ACTION_UP} = up, {ACTION_DOWN} = down)")

ALE action would be: 3 (2 = up, 3 = down)


Three steps from a probability to a controller input: the network produces a number in
$(0,1)$, a Bernoulli draw turns it into 0 or 1, and a lookup turns that into ALE action 2 or
3. Those constants come from `pong.env`, so this cannot disagree with what training used.

`nn.Sequential` above is shorthand for a stack of layers with nothing clever between them —
equivalent to the class you wrote in Worksheet 2, with less typing and less flexibility.

## 8. Grad accumulation (concept)

If you call `backward()` several times without `zero_grad()`, gradients add up. That is useful for “update once every $N$ episodes.” Pattern:

```text
zero_grad
for episode in 1..N:
    ... compute loss ...
    loss.backward()      # accumulates
optimizer.step()
zero_grad
```

In [14]:
torch.manual_seed(3)
model = TinyBinaryNet()
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-2)

optimizer.zero_grad()
for _ in range(3):
    p = model(torch.randn(8))
    lp = Bernoulli(probs=p).log_prob(Bernoulli(probs=p).sample())
    loss = -lp * 1.0
    loss.backward()                      # accumulate

grad_norm = model.layer2.weight.grad.norm().item()
optimizer.step()
optimizer.zero_grad()
print("grad norm after 3 accumulated backwards:", grad_norm)

grad norm after 3 accumulated backwards: 1.2241190671920776


In [15]:
# The single number above is the end state. Watch it build up instead.
torch.manual_seed(3)
watch_model = TinyBinaryNet()
watch_optimizer = torch.optim.RMSprop(watch_model.parameters(), lr=1e-2)

watch_optimizer.zero_grad()
for episode in range(3):
    probability = watch_model(torch.randn(8))
    distribution = Bernoulli(probs=probability)
    loss = -distribution.log_prob(distribution.sample())
    loss.backward()                       # accumulates, does not replace
    norm = watch_model.layer2.weight.grad.norm().item()
    print(f"after backward {episode + 1}: grad norm = {norm:.5f}")

watch_optimizer.step()
watch_optimizer.zero_grad()
print("after step + zero_grad:", watch_model.layer2.weight.grad)


after backward 1: grad norm = 0.68542
after backward 2: grad norm = 1.18887
after backward 3: grad norm = 1.22412
after step + zero_grad: None


The three norms are the evidence. They are not three independent measurements: each is the
norm of the **running sum**, so the value grows as more episodes are added. If `zero_grad()`
were being called inside the loop you would see three unrelated numbers instead of a
sequence that builds.

After `step()` and `zero_grad()`, the gradient is gone — back to `None`, ready for the next
batch.

One consequence worth carrying to the real loop: because gradients are cleared immediately
after each step, a checkpoint saved just after a step never contains a half-finished
gradient. That is precisely what makes it safe to stop training and resume exactly — the
property `scripts/train.py` relies on.


## 9. Checkpoint after an update

In [16]:
ckpt = pong.PROJECT_ROOT / "assets" / "_demo_train_step.pt"   # not Path("assets"): the notebook is not the root
torch.save(
    {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    },
    ckpt,
)
print("saved", ckpt)

saved /Users/piyush/Code/ml/pong-pytorch/assets/_demo_train_step.pt


A checkpoint is a plain dictionary and you decide what goes in it. Here: the weights and the
optimizer state.

Saving the optimizer matters more than it looks. RMSprop keeps that running average of
squared gradients; throw it away and a resumed run spends a while recovering its footing,
taking oversized steps exactly as you saw in section 4.

The real `scripts/train.py` saves two more things — the episode count and the settings that
produced the run — because without them a checkpoint cannot say where it had got to or what
it was.

## 10. What you can do now

| Skill | API |
|---|---|
| Sample a binary choice | `Bernoulli(probs=p).sample()` |
| Score it | `.log_prob(action)` |
| Scalar objective | `-(log_probs * weights).sum()` |
| Update weights | `backward` → `optimizer.step` → `zero_grad` |
| Turn rewards into weights | `discount_rewards` (+ normalize) |

Still ahead (later notebook / main project): turn pixels into the input vector, step a real environment, and run this loop for many episodes.

You now have the train-step core: probability → sample → log_prob → weighted loss → optimizer.

## 11. Your exercises

In [17]:
# Exercise 3.1 — Verify log_prob by hand.
# For p = 0.3, compute log_prob for BOTH possible actions with Bernoulli, then check
# each against math.log of the probability you expect. Print them side by side.

# your code here

**Exercise 3.2.** Explain in your own words why a *negative* weight pushes an action's
probability down, referring to the sign of `log_prob` and the minus sign in front of the
sum. Work one entry through numerically.

*Your written answer here.*

In [18]:
# Exercise 3.3 — How far does credit reach?
# For gamma in (0.9, 0.99, 0.999), compute how many steps back credit must travel before
# it falls below 0.01 of the original reward. Print a small table. Then say which gamma
# suits a game where a point takes ~200 frames to develop.

# your code here

In [19]:
# Exercise 3.4 — Show the leak.
# Build a reward list with two rallies, e.g. [0,0,0,1, 0,0,0,-1]. Print plain vs
# reset-on-score returns, identify the exact positions where they differ, and explain
# in a comment why the plain version is wrong there.

# your code here

In [20]:
# Exercise 3.5 — Does accumulation matter?
# Run the grad-accumulation loop twice: once WITH zero_grad() inside the loop and once
# without. Print the gradient norm just before optimizer.step() in each case. Which is
# larger, and why?

# your code here

**Exercise 3.6.** Normalising returns turned some positive weights negative. Is that
desirable in an episode the agent *won* 21–0? Argue both sides, then say what you would do.

*Your written answer here.*

In [21]:
# Exercise 3.7 — Make it learn something checkable.
# Train TinyBinaryNet on a rule you invent: reward +1 when the sampled action is 1 and
# the first element of x is positive, else -1. Run a few hundred updates and print the
# fraction of correct actions every 50. Does it improve? This is the smallest end-to-end
# check that your loop learns at all.

# your code here

## 12. Questions to think about

1. The loss is unbounded and can be negative, so "loss = -112" says nothing about how well
   the agent plays. What would you monitor instead, and why does `scripts/train.py` track a
   running mean of episode reward?

2. Normalising makes credit relative to the episode's own average. What happens in an
   episode where every action really was bad?

3. We reset the running return at every scored point. Are there other places in Pong where
   credit should not flow that we are *not* cutting?

4. Batching over 10 episodes reduces noise, but the policy being updated is 10 episodes
   stale by the end of the batch. Why is that acceptable? When might it not be?

5. The update in section 4 made a sampled action more likely purely because we said the
   weight was 1.0. Given the agent starts knowing nothing and acts randomly, why does this
   converge on good play rather than reinforcing early luck?

6. Every piece here is a few lines of code, yet the real thing needs around 99,000 episodes
   and days of CPU. Where does that time go, and what would you measure to find out?

## 13. What you have learnt

| You can now | API |
|---|---|
| Turn a probability into a choice | `Bernoulli(probs=p).sample()` |
| Score the choice you made | `.log_prob(action)`, and why it is always negative |
| Build a scalar objective | `-(log_probs * weights).sum()`, and what the sign of $w$ does |
| Run one update | `zero_grad` → `backward` → `step`, and check the weights moved |
| Turn rewards into weights | `discount_rewards`, and what $\gamma$ actually controls |
| Make credit relative | `normalize`, and why it creates negative weights |
| Stop credit leaking | reset the running return at a scored point |
| Reduce update noise | accumulate gradients over several episodes |
| Save properly | model **and** optimizer state |

Put the three worksheets together and the whole agent is: preprocess a frame to 6400
numbers, take the difference between consecutive frames so motion is visible, feed it to a
6400 → 200 → 1 network, sample an action, store the log-prob, collect rewards, discount with
resets at each point, normalise, accumulate over 10 episodes, step.

That is `scripts/train.py`. Everything else is waiting.

**Next:** `04_watch_agent_worksheet.ipynb` loads a policy that has already done all that.